# Chapter 3 · Productionize on Databricks

Take the **exact** `OTel-Embedding-335M` model from [Chapters 0–1](./01_otel_rag_pipeline.ipynb) and put it on the platform: **log → Unity Catalog → Model Serving → Vector Search**. What the model *does* is unchanged; we're just governing, serving, and scaling it.

**Prerequisites:** a Databricks workspace, a Unity Catalog you can create schemas/models in, Model Serving entitlement, and a Vector Search endpoint (or permission to create one).

> ⚠️ This notebook **creates billable resources** (a serving endpoint + a Vector Search index). Run it deliberately, and clean up (final cell) when done.

> OTel = **Open Telco**, not OpenTelemetry.

In [ ]:
%pip install -q "sentence-transformers>=3.0.0" "mlflow>=2.13" databricks-vectorsearch databricks-sdk
dbutils.library.restartPython()

## 0. Config — edit for your workspace

In [ ]:
CATALOG      = "main"                 # a UC catalog you can write to (e.g. your_catalog)
SCHEMA       = "otel_selfhealing"     # schema for the demo assets
VS_ENDPOINT  = "otel_vs_endpoint"     # a Vector Search endpoint (created below if missing)

EMB_HF_ID    = "farbodtavakkoli/OTel-Embedding-335M"
EMB_UC_MODEL = f"{CATALOG}.{SCHEMA}.otel_embedding_335m"
EMB_ENDPOINT = "otel-embedding-335m"
EMB_DIM      = 1024                   # BGE-large output dimension

CORPUS_TABLE = f"{CATALOG}.{SCHEMA}.otel_corpus"
INDEX_NAME   = f"{CATALOG}.{SCHEMA}.otel_corpus_index"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")   # assumes the catalog already exists
print("config ok ->", EMB_UC_MODEL)

## 1. Log + register the embedding model to Unity Catalog

`mlflow.sentence_transformers` is the right flavor for this model. Registering to UC makes it a **governed, versioned** asset — the foundation everything else builds on.

In [ ]:
import mlflow
import numpy as np
from sentence_transformers import SentenceTransformer
from mlflow.models.signature import infer_signature

mlflow.set_registry_uri("databricks-uc")

model = SentenceTransformer(EMB_HF_ID)
example = ["low 5G downlink throughput at low PRB load"]
sig = infer_signature(example, model.encode(example, normalize_embeddings=True))

with mlflow.start_run(run_name="log-otel-embedding-335m"):
    info = mlflow.sentence_transformers.log_model(
        model, artifact_path="model",
        signature=sig, input_example=example,
        registered_model_name=EMB_UC_MODEL,
    )
print("registered:", EMB_UC_MODEL, "->", info.model_uri)

In [ ]:
from mlflow.tracking import MlflowClient
c = MlflowClient(registry_uri="databricks-uc")
EMB_VERSION = max(int(mv.version) for mv in c.search_model_versions(f"name='{EMB_UC_MODEL}'"))
print("latest version:", EMB_VERSION)

## 2. Serve it — a CPU Model Serving endpoint

335M serves comfortably on **CPU**, so no GPU is required. Scale-to-zero keeps it cheap between demos. (Idempotent: creates the endpoint, or updates it to the new version if it already exists.)

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

w = WorkspaceClient()
served = [ServedEntityInput(entity_name=EMB_UC_MODEL, entity_version=str(EMB_VERSION),
                            scale_to_zero_enabled=True, workload_size="Small", workload_type="CPU")]
cfg = EndpointCoreConfigInput(served_entities=served)

existing = [e.name for e in w.serving_endpoints.list()]
if EMB_ENDPOINT in existing:
    w.serving_endpoints.update_config_and_wait(name=EMB_ENDPOINT, served_entities=served)
    print("updated endpoint:", EMB_ENDPOINT)
else:
    w.serving_endpoints.create_and_wait(name=EMB_ENDPOINT, config=cfg)
    print("created endpoint:", EMB_ENDPOINT)

In [ ]:
# Call the live endpoint
import mlflow.deployments
client = mlflow.deployments.get_deploy_client("databricks")
resp = client.predict(endpoint=EMB_ENDPOINT, inputs={"inputs": ["PCI collision between neighbor cells"]})
print("embedding shape:", np.array(resp["predictions"]).shape)

## 3. Build the Vector Search index

Write the telecom corpus to a **CDF-enabled Delta table** with **L2-normalized** embeddings — the Chapter 1 lesson: Databricks Vector Search ranks by L2 distance only, so normalizing makes L2 ranking equal cosine ranking. Then build a **Delta-Sync** index.

In [ ]:
import pandas as pd

CORPUS = [
    {"cite": "3GPP TS 38.214 5.2", "text": "Low SINR/CQI forces a lower MCS, reducing per-UE throughput even at moderate PRB. Low throughput with LOW PRB load indicates a radio-quality problem, not congestion."},
    {"cite": "3GPP TS 36.213 7.2", "text": "LTE downlink throughput saturates as PRB utilization approaches 100%. Sustained PRB above 90% across neighboring cells in the busy hour is the signature of CONGESTION."},
    {"cite": "O-RAN WG1 UC",      "text": "Congestion remediation order: load-balance to under-utilized neighbors, enable carrier aggregation, add carrier/spectrum, then cell split or new site."},
    {"cite": "3GPP TS 36.331 8.1", "text": "PCI collision between neighbor cells corrupts measurement reports and handovers, degrading SINR. Resolve PCI conflicts before RF optimization."},
    {"cite": "RF Ops Playbook",    "text": "Correlate recurring EXTERNAL_INTERFERENCE_UL alarms with low-SINR cells before adjusting antenna tilt or transmit power."},
    {"cite": "O-RAN F1",           "text": "The F1 interface connects the O-RAN Distributed Unit (O-DU) to the O-RAN Central Unit (O-CU), carrying F1-C and F1-U traffic."},
    {"cite": "CPRI/eCPRI Ops",     "text": "Loss of the CPRI/eCPRI fronthaul link between radio unit and baseband unit takes the cell off the air; dispatch a technician to inspect the fiber path."},
    {"cite": "TM Forum Open API",  "text": "Standardized management interfaces enable vendor-agnostic data collection across Huawei/Ericsson/Nokia OSS/BSS for closed-loop automation."},
]
texts = [d["text"] for d in CORPUS]
vecs  = model.encode(texts, normalize_embeddings=True)   # L2-normalized (unit vectors)

pdf = pd.DataFrame({"doc_id": [f"d{i}" for i in range(len(CORPUS))],
                    "cite":   [d["cite"] for d in CORPUS],
                    "content": texts,
                    "embedding": [v.tolist() for v in vecs]})
(spark.createDataFrame(pdf).write.mode("overwrite")
      .option("delta.enableChangeDataFeed", "true").saveAsTable(CORPUS_TABLE))
spark.sql(f"ALTER TABLE {CORPUS_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
print("wrote", CORPUS_TABLE, "rows:", len(CORPUS))

In [ ]:
from databricks.vector_search.client import VectorSearchClient
vsc = VectorSearchClient(disable_notice=True)

# Ensure the Vector Search endpoint exists (creation can take a few minutes the first time).
try:
    vsc.get_endpoint(VS_ENDPOINT)
    print("using existing VS endpoint:", VS_ENDPOINT)
except Exception:
    vsc.create_endpoint(VS_ENDPOINT, endpoint_type="STANDARD")
    print("creating VS endpoint:", VS_ENDPOINT, "(wait until ONLINE, then re-run this cell)")

In [ ]:
# Self-managed Delta-Sync index over the pre-computed (normalized) embedding column.
index = vsc.create_delta_sync_index(
    endpoint_name=VS_ENDPOINT,
    index_name=INDEX_NAME,
    source_table_name=CORPUS_TABLE,
    pipeline_type="TRIGGERED",
    primary_key="doc_id",
    embedding_dimension=EMB_DIM,
    embedding_vector_column="embedding",
)
print("index creating:", INDEX_NAME, "(TRIGGERED sync; wait for ONLINE)")

In [ ]:
# Query with a NORMALIZED query vector (the other half of the Chapter 1 lesson).
q = model.encode("two adjacent cells were configured with the same PCI", normalize_embeddings=True).tolist()
idx = vsc.get_index(VS_ENDPOINT, INDEX_NAME)
res = idx.similarity_search(query_vector=q, columns=["cite", "content"], num_results=3)
for row in res["result"]["data_array"]:
    print(row)

## 4. Reranker + LLM follow the same pattern

Same log → UC → serve loop, different flavor and workload:

```python
# Reranker (OTel-Reranker-0.6B, cross-encoder) — mlflow.transformers, small GPU serving
mlflow.transformers.log_model(
    transformers_model={"model": rk_model, "tokenizer": rk_tok},
    artifact_path="model", task="text-classification",
    registered_model_name=f"{CATALOG}.{SCHEMA}.otel_reranker_06b",
    metadata={"task": "llm/v1/..."})            # workload_type="GPU_SMALL"

# LLM (e.g. OTel-LLM-20B, gpt-oss base) — provisioned throughput IS supported for this base arch
mlflow.transformers.log_model(..., task="llm/v1/chat",
    registered_model_name=f"{CATALOG}.{SCHEMA}.otel_llm_20b")
# then serve with provisioned throughput (min/max tps) instead of a fixed workload size
```

**Reality check (from Chapter 1):** validate the reranker actually *discriminates* before wiring it in — in production this exact reranker sometimes returned near-constant scores and had to be disabled in favor of an LLM reranker.

## What's next

The model is now **governed** (UC), **served** (REST endpoint), and **indexed** (Vector Search). [Chapter 4](../chapters/04-govern-capture/README.md) adds the part most demos skip: **capture every inference** in an inference table, **monitor** it, and track **cost economics** (open-source in-zone vs. frontier).

---
### Cleanup (optional — stops billing on the demo resources)

In [ ]:
# Uncomment to tear down the endpoint + index created above.
# w.serving_endpoints.delete(EMB_ENDPOINT)
# vsc.delete_index(VS_ENDPOINT, INDEX_NAME)
# print("cleaned up")